# Predictor Scatter

Generates a grid of scatter plots comparing SGE fitness score (x-axis) against available computational predictors (y-axis): AlphaMissense (`am_score`), REVEL (`revel_score`), CADD (`cadd_score`), MutPred2 (`MutPred2`), and SpliceAI (`max_SpliceAI`). SGE thresholds and published predictor cut-offs are overlaid as reference lines.

**Required data:** `scores` and `thresholds` sheets from the pipeline output Excel file (`*_data.xlsx`). At least one predictor column must be present in `scores`.

**Required packages:** `sge-utils` (SimpleSGEViz) installed in the active environment.

**Saving:** PNG/SVG require `vl-convert-python` (`pip install vl-convert-python`). Change the extension to `.html` for interactive output (no extra package needed).

In [ ]:
import pandas as pd
from sgeviz.figures import predictor_scatter

In [ ]:
# --- Configuration ---
excel_path = "GENE_data.xlsx"   # path to pipeline output Excel file
gene = "GENE"                   # gene name for plot title
output_path = f"{gene}_predictor_scatter.png"  # change to .html for interactive output, or .svg

# --- Plot customization (optional) ---
panel_width = 350    # width of each scatter panel in pixels
panel_height = 250   # height of each scatter panel in pixels

In [ ]:
# --- Load data ---
scores_df = pd.read_excel(excel_path, sheet_name="scores")
thresh_df = pd.read_excel(excel_path, sheet_name="thresholds")

thresholds = [
    thresh_df["non_functional_threshold"].iloc[0],
    thresh_df["functional_threshold"].iloc[0],
]

# Coerce predictor columns to numeric (pipeline may store missing values as '-' strings)
predictor_cols = ["am_score", "revel_score", "cadd_score", "MutPred2", "max_SpliceAI"]
for col in predictor_cols:
    if col in scores_df.columns:
        scores_df[col] = pd.to_numeric(scores_df[col], errors="coerce")

print(f"Loaded {len(scores_df)} variants. Thresholds: {thresholds}")

In [ ]:
# --- Generate plot ---
chart = predictor_scatter.make_plot(scores_df, thresholds, gene=gene, panel_width=panel_width, panel_height=panel_height)

if chart is None:
    print("No predictor columns found in scores sheet. Add am_score, revel_score, cadd_score, MutPred2, or max_SpliceAI columns.")
else:
    chart

In [ ]:
# --- Save ---
if chart is not None:
    chart.save(output_path)
    print(f"Saved: {output_path}")